[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 协同过滤：重构代码
-----

在这个实操里，你需要重构课上看到的代码，以便处理 [Movielens 1M 数据集](https://grouplens.org/datasets/movielens/1m/)


## 1. 准备工作


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os.path as op
import imp
import numpy as np

from zipfile import ZipFile
try:
    from urllib.request import urlretrieve
except ImportError:  # except ImportError:  # Python 2 兼容
    from urllib import urlretrieve

# 这一行需要修改：
data_folder = '/content/'


ML_1M_URL = "http://files.grouplens.org/datasets/movielens/ml-1m.zip"
ML_1M_FILENAME = op.join(data_folder,ML_1M_URL.rsplit('/', 1)[1])
ML_1M_FOLDER = op.join(data_folder,'ml-1m')

In [ ]:
if not op.exists(ML_1M_FILENAME):
    print('Downloading %s to %s...' % (ML_1M_URL, ML_1M_FILENAME))
    urlretrieve(ML_1M_URL, ML_1M_FILENAME)

if not op.exists(ML_1M_FOLDER):
    print('Extracting %s to %s...' % (ML_1M_FILENAME, ML_1M_FOLDER))
    ZipFile(ML_1M_FILENAME).extractall(data_folder)

## 2. 数据分析与格式化


和课上一样，我们先用 [Python 数据分析库](http://pandas.pydata.org/) 加载数据


In [ ]:
import pandas as pd
all_ratings = pd.read_csv(op.join(ML_1M_FOLDER, 'ratings.dat'), sep='::',
                          names=["user_id", "item_id", "ratings", "timestamp"],engine='python')
all_ratings.head()

In [ ]:
list_movies_names = []
list_item_ids = []
with open(op.join(ML_1M_FOLDER, 'movies.dat'), encoding = "ISO-8859-1") as fp:
    for line in fp:
        list_item_ids.append(line.split('::')[0])
        list_movies_names.append(line.split('::')[1])
        
movies_names = pd.DataFrame(list(zip(list_item_ids, list_movies_names)), 
               columns =['item_id', 'item_name']) 
movies_names.head()

这里我们把电影标题加到数据里。


In [ ]:
movies_names['item_id']=movies_names['item_id'].astype(int)
all_ratings['item_id']=all_ratings['item_id'].astype(int)
all_ratings = all_ratings.merge(movies_names,on='item_id')

In [ ]:
all_ratings.head()

数据框 `all_ratings` 包含我们问题的所有原始数据。


In [ ]:
#条目数量
len(all_ratings)

In [ ]:
all_ratings['ratings'].describe()

In [ ]:
all_ratings['ratings'].unique()

In [ ]:
all_ratings['user_id'].describe()

In [ ]:
# 不同用户的数量
total_user_id = len(all_ratings['user_id'].unique())
print(total_user_id)

可以看到，和课上一样，用户的索引似乎是从 1 到 6040。下面验证一下。


In [ ]:
list_user_id = list(all_ratings['user_id'].unique())
list_user_id.sort()

In [ ]:
for i,j in enumerate(list_user_id):
    if j != i+1:
        print(i,j) 

我们创建一个新列 `user_num`，让用户的索引从 0 到 6039：


In [ ]:
all_ratings['user_num'] = all_ratings['user_id'].apply(lambda x :x-1)

In [ ]:
all_ratings.head()

现在我们看看电影。


In [ ]:
all_ratings['item_id'].describe()

In [ ]:
# 被评过分的不同物品数量
total_item_id = len(all_ratings['item_id'].unique())
print(total_item_id)

这里有一个明显的问题：有 3706 部不同的电影，但 `item_id` 的范围从 1 到 3952。所以中间有空缺。因此你需要做的第一件事是创建一个新列 `item_num`，让所有电影的索引从 0 到 3705。


In [ ]:
#
# 你的代码
#

这个函数会验证你的结果是否正确。


In [ ]:
def check_ratings_num(df):
    item_num = set(df['item_num'])
    if item_num == set(range(len(item_num))):
        return True
    else:
        return False

In [ ]:
check_ratings_num(all_ratings)

In [ ]:
all_ratings.head()

现在我们将使用 [scikit-learn](http://scikit-learn.org/stable/) 里一个现成的函数，把数据分成 _train_、_val_ 和 _test_。


In [ ]:
from sklearn.model_selection import train_test_split

ratings_trainval, ratings_test = train_test_split(all_ratings, test_size=0.1, random_state=42)

ratings_train, ratings_val = train_test_split(ratings_trainval, test_size=0.1, random_state=42)

## 3. 模型

现在我们要稍微修改一下课上看到的 `FactorizationModel` 类。内部我们仍然使用 `Model_dot`，但现在改用 PyTorch 的 dataloader。


In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
def df_2_tensor(df, device):
    # 从数据框返回 user_num、item_num、rating 三元组
    user_num = np.asarray(df['user_num'])
    item_num = np.asarray(df['item_num'])
    rating = np.asarray(df['ratings'])
    return torch.from_numpy(user_num).to(device), torch.from_numpy(item_num).to(device), torch.from_numpy(rating).to(device)

下面，我们构造 3 个张量，分别包含训练集的 `user_num`、`item_num` 和 `rating`。所有张量的形状相同，因此 `train_user_num[i]` 看过 `train_item_num[i]` 并给出了 `train_rating[i]` 的评分。


In [ ]:
train_user_num, train_item_num, train_rating = df_2_tensor(ratings_train,device)

现在对验证集和测试集做同样的事。


In [ ]:
val_user_num, val_item_num, val_rating = df_2_tensor(ratings_val,device)
test_user_num, test_item_num, test_rating = df_2_tensor(ratings_test,device)

下面的代码取自课程


In [ ]:
class ScaledEmbedding(nn.Embedding):
    """
    Embedding layer that initialises its values
    to using a normal variable scaled by the inverse
    of the embedding dimension.
    """
    def reset_parameters(self):
        """
        Initialize parameters.
        """

        self.weight.data.normal_(0, 1.0 / self.embedding_dim)
        if self.padding_idx is not None:
            self.weight.data[self.padding_idx].fill_(0)


class ZeroEmbedding(nn.Embedding):
    """
    Used for biases.
    """

    def reset_parameters(self):
        """
        Initialize parameters.
        """

        self.weight.data.zero_()
        if self.padding_idx is not None:
            self.weight.data[self.padding_idx].fill_(0)

In [ ]:
class DotModel(nn.Module):
    
    def __init__(self,
                 num_users,
                 num_items,
                 embedding_dim=32):
        
        super(DotModel, self).__init__()
        
        self.embedding_dim = embedding_dim
        
        self.user_embeddings = ScaledEmbedding(num_users, embedding_dim)
        self.item_embeddings = ScaledEmbedding(num_items, embedding_dim)
        self.user_biases = ZeroEmbedding(num_users, 1)
        self.item_biases = ZeroEmbedding(num_items, 1)
                
        
    def forward(self, user_ids, item_ids):
        #
        # 你的代码
        #


In [ ]:
net = DotModel(total_user_id,total_item_id).to(device)

现在在一个小 batch 上测试你的网络。


In [ ]:
predictions = net(train_user_num[:5], train_item_num[:5])
predictions

In [ ]:
def regression_loss(predicted_ratings, observed_ratings):
    return ((observed_ratings - predicted_ratings) ** 2).mean()

In [ ]:
loss_fn = regression_loss
loss = loss_fn(predictions, train_rating[:5])
loss

现在你需要构造一个数据集和一个 dataloader。为此，你可以先定义一个函数：接收上面定义的张量作为参数，返回一个列表；然后再定义一个函数：接收数据集、batchsize 和一个布尔值（是否打乱）作为参数。我们不再使用课上用的 `shuffle` 和 `minibatch` 函数。


In [ ]:
def tensor_2_dataset(user,item,rating):
    # 在这里写你的代码
    
def make_dataloader(dataset,bs,shuffle):
    # 在这里写你的代码
    

In [ ]:
train_dataset = tensor_2_dataset(train_user_num,train_item_num, train_rating)
val_dataset = tensor_2_dataset(val_user_num,val_item_num,val_rating)
test_dataset = tensor_2_dataset(test_user_num, test_item_num, test_rating)

In [ ]:
train_dataloader = make_dataloader(train_dataset,1024,True)
val_dataloader = make_dataloader(val_dataset,1024, False)
test_dataloader = make_dataloader(test_dataset,1024,False)

这里你需要修改课上看到的代码：
 - 去掉 init 里的 batch_size
 - fit 函数现在应该接收训练 dataloader 和验证 dataloader 作为参数。每个 epoch 结束时，你在验证集上运行 test 方法。然后同时打印训练集和验证集上的损失，看看是否过拟合。


In [ ]:
class FactorizationModel(object):
    
    def __init__(self, embedding_dim=32, n_iter=10, l2=0.0,
                 learning_rate=1e-2, device=device, net=None, num_users=None,
                 num_items=None,random_state=None):
        
        self._embedding_dim = embedding_dim
        self._n_iter = n_iter
        self._learning_rate = learning_rate
        self._l2 = l2
        self._device = device
        self._num_users = num_users
        self._num_items = num_items
        self._net = net
        self._optimizer = None
        self._loss_func = None
        self._random_state = random_state or np.random.RandomState()
             
        
    def _initialize(self):
        if self._net is None:
            self._net = DotModel(self._num_users, self._num_items, self._embedding_dim).to(self._device)
        
        self._optimizer = optim.Adam(
                self._net.parameters(),
                lr=self._learning_rate,
                weight_decay=self._l2
            )
        
        self._loss_func = regression_loss
        
    
    @property
    def _initialized(self):
        return self._optimizer is not None
    
    def __repr__(self):
        return _repr_model(self)
    
    def fit(self, dataloader, val_dataloader, verbose=True):       
        if not self._initialized:
            self._initialize()
            
        for epoch_num in range(self._n_iter):
            epoch_loss = 0.0
            self._net.train(True)

            #
            # 你的代码
            #
                
            
            epoch_loss = epoch_loss / (minibatch_num + 1)
            loss_test = self.test(val_dataloader)

            if verbose:
                print('Epoch {}: loss_train {}, loss_val {}'.format(epoch_num, epoch_loss,loss_test))
        
            if np.isnan(epoch_loss) or epoch_loss == 0.0:
                raise ValueError('Degenerate epoch loss: {}'
                                 .format(epoch_loss))
    
    
    def test(self,dataloader, verbose = False):
        self._net.train(False)
        L1loss = torch.nn.L1Loss()
        test_loss = 0.0
        test_mae = 0.0
        #
        # 在这里写你的代码（mae = 平均绝对误差）
        #
                
        test_loss = test_loss / (minibatch_num + 1)
        test_mae = test_mae / (minibatch_num+1)
        if verbose:
            print(f"RMSE: {np.sqrt(test_loss)}, MAE: {test_mae}")
        return loss.item()

In [ ]:
model = FactorizationModel(embedding_dim=50,  # model = FactorizationModel(embedding_dim=50,  # 潜变量维度
                                   n_iter=5,  # n_iter=5,  # 训练的 epoch 数
                                   learning_rate=5e-4,
                                   l2=1e-8,  # l2=1e-8,  # L2 正则化强度
                                   num_users=total_user_id,
                                   num_items=total_item_id)

In [ ]:
model.fit(train_dataloader,val_dataloader)

In [ ]:
_= model.test(test_dataloader,True)

试着调整参数，打败[这里](https://github.com/NicolasHug/Surprise)给出的基准结果


## 4. 最好和最差的电影

现在你需要按偏置对电影排序。为此，你需要取出电影的偏置，做一份 `[电影名, 它的偏置]` 二元组的列表，然后按偏置排序。你可以用列表的 sort 方法。


## 5. 电影 embedding 的 PCA

现在你还可以玩玩你的算法为电影学到的 embedding。


In [ ]:
from sklearn.decomposition import PCA
from operator import itemgetter

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)